# Notebook 04: Flax NNX Fundamentals

## Why This Matters

JAX gives you automatic differentiation and compilation primitives, but managing model parameters manually gets tedious. Flax NNX is Google DeepMind's recommended neural network library for JAX. It powers Gemma, Scenic, and much of DeepMind's research infrastructure. Unlike PyTorch where parameters live inside the module, Flax NNX makes parameters explicit, which is philosophically JAX-aligned and makes sharding across devices natural.


In [ ]:
%pip install -q jax jaxlib flax optax numpy

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import numpy as np
print("Flax version:", __import__("flax").__version__)
print("JAX version:", jax.__version__)
print("Ready.")

## Background: NNX vs Linen

Flax has two APIs:

**Linen** (older, ~2020): fully functional. Modules have no state; you call `model.init(key, x)` to get a params dict, then `model.apply(params, x)` to run. All state is external to the module. Very pure, but verbose.

**NNX** (newer, ~2024): stateful-ish. Modules hold their parameters as attributes (`nnx.Param`). Feels more like PyTorch. Under the hood, NNX uses JAX's pytree system to extract/inject state cleanly. NNX is now the recommended API.

**Key difference from PyTorch**: Even in NNX, parameters are still plain JAX arrays -- they just live as module attributes. The `nnx.state()` function extracts them all as a pytree for use with `jax.jit` and `jax.grad`.


In [ ]:
# nnx.Module: defining a custom layer
class LinearLayer(nnx.Module):
    def __init__(self, in_features, out_features, rngs: nnx.Rngs):
        # Parameters are nnx.Param -- they get tracked by NNX
        self.weight = nnx.Param(jax.random.normal(rngs.params(), (in_features, out_features)) * 0.01)
        self.bias   = nnx.Param(jnp.zeros(out_features))
        self.in_features = in_features
        self.out_features = out_features
    
    def __call__(self, x):
        return x @ self.weight.value + self.bias.value

rngs = nnx.Rngs(0)  # seed 0 for parameter initialization
layer = LinearLayer(4, 8, rngs)

x = jnp.ones((2, 4))
y = layer(x)
print('Output shape:', y.shape)
print('Weight shape:', layer.weight.value.shape)
print('Bias shape:  ', layer.bias.value.shape)

## 2. Built-in Layers: nnx.Linear, nnx.Conv, nnx.BatchNorm

NNX provides standard layers. The `rngs` argument threads random keys for parameter initialization and stochastic operations (dropout). This is JAX's explicit randomness philosophy applied to modules.


In [ ]:
# Built-in NNX layers
rngs = nnx.Rngs(params=0, dropout=1)  # separate keys for params and dropout

# Linear layer
linear = nnx.Linear(16, 32, rngs=rngs)
x = jnp.ones((4, 16))
print('Linear output:', linear(x).shape)

# Conv layer (expects NHWC by default)
conv = nnx.Conv(in_features=3, out_features=16, kernel_size=(3, 3), rngs=rngs)
img = jnp.ones((2, 28, 28, 3))  # batch, H, W, channels
print('Conv output:  ', conv(img).shape)

# BatchNorm (needs 'use_running_average' argument at call time)
bn = nnx.BatchNorm(num_features=32, rngs=rngs)
x_bn = jnp.ones((4, 32))
print('BatchNorm output:', bn(x_bn, use_running_average=False).shape)

# Dropout
dropout = nnx.Dropout(rate=0.5, rngs=rngs)
x_drop = jnp.ones((4, 16))
# During training: deterministic=False
# During inference: deterministic=True (no dropout applied)
out_train = dropout(x_drop, deterministic=False)
out_infer = dropout(x_drop, deterministic=True)
print('Dropout train (some zeros):', (out_train == 0).sum())
print('Dropout infer (no zeros):  ', (out_infer == 0).sum())

## 3. Extracting Parameters with nnx.state()

The key NNX operation: `nnx.state(model)` extracts all parameters as a pytree (nested dict of JAX arrays). This is what you pass to `jax.grad` and what gets updated during training.

This explicit parameter extraction is what makes JAX training loops look different from PyTorch -- there is no `.parameters()` iterator; instead you extract the entire state tree at once.


In [ ]:
# nnx.state: extract parameters for use with jax.grad
rngs = nnx.Rngs(0)
linear = nnx.Linear(4, 8, rngs=rngs)

# Extract all trainable state
state = nnx.state(linear)
print('State type:', type(state))
print('State keys:', list(state.keys()))
print('Kernel shape:', state['kernel'].value.shape)
print('Bias shape:  ', state['bias'].value.shape)

# Count parameters
def count_params(model):
    state = nnx.state(model)
    return sum(v.value.size for v in jax.tree_util.tree_leaves(state))

print('\nParameter count:', count_params(linear))

# nnx.graphdef + nnx.state: separate structure from values (for serialization)
graphdef, state = nnx.split(linear)
print('\nGraphdef type:', type(graphdef))
print('State type:', type(state))

# Reconstruct from graphdef + state
linear_reconstructed = nnx.merge(graphdef, state)
x = jnp.ones((2, 4))
print('Outputs match after split/merge:', jnp.allclose(linear(x), linear_reconstructed(x)))

## 4. Building a Multi-Layer Perceptron with NNX

Let us build a complete MLP with multiple layers, batch normalization, and dropout. This is the canonical NNX pattern you will use as a starting point for most models.


In [ ]:
class MLP(nnx.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout_rate, rngs: nnx.Rngs):
        self.layers = []
        dims = [input_dim] + list(hidden_dims)
        for i in range(len(dims) - 1):
            self.layers.append(nnx.Linear(dims[i], dims[i+1], rngs=rngs))
        self.output_layer = nnx.Linear(dims[-1], output_dim, rngs=rngs)
        self.dropout = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.bn_layers = [nnx.BatchNorm(d, rngs=rngs) for d in hidden_dims]
    
    def __call__(self, x, training: bool = True):
        for layer, bn in zip(self.layers, self.bn_layers):
            x = layer(x)
            x = bn(x, use_running_average=not training)
            x = jax.nn.relu(x)
            x = self.dropout(x, deterministic=not training)
        return self.output_layer(x)

rngs = nnx.Rngs(params=0, dropout=1)
model = MLP(
    input_dim=784,
    hidden_dims=[256, 128],
    output_dim=10,
    dropout_rate=0.3,
    rngs=rngs
)

# Count parameters
def count_params(model):
    state = nnx.state(model)
    total = 0
    for leaf in jax.tree_util.tree_leaves(state):
        if hasattr(leaf, 'value'):
            total += leaf.value.size
    return total

n_params = count_params(model)
print(f'MLP parameter count: {n_params:,}')

# Forward pass
key = jax.random.PRNGKey(42)
x = jax.random.normal(key, (32, 784))

logits_train = model(x, training=True)
logits_infer = model(x, training=False)
print('Logits shape (train):', logits_train.shape)
print('Logits shape (infer):', logits_infer.shape)
print('Train != infer (due to dropout):', not jnp.allclose(logits_train, logits_infer))

## Summary

### Key Concepts

| Concept | NNX API | Notes |
|---------|---------|-------|
| Define module | `class M(nnx.Module)` | Parameters as `nnx.Param` attributes |
| Initialize | `model = M(..., rngs=nnx.Rngs(seed))` | Explicit RNG key |
| Forward pass | `model(x)` | Standard Python call |
| Extract state | `nnx.state(model)` | Returns pytree of parameters |
| Split/merge | `nnx.split(model)` / `nnx.merge(gd, state)` | For serialization |
| Built-in layers | `nnx.Linear`, `nnx.Conv`, `nnx.BatchNorm`, `nnx.Dropout` | Standard toolkit |

**Next**: `05_training_with_optax.ipynb` -- optimizers, learning rate schedules, and a full training loop.
